<a href="https://colab.research.google.com/github/robertbarcik/vector_databases-tutorial/blob/main/2.%20Pinecone/1_Managed_Vector_Database_Pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Managed Vector Database: Pinecone

So far the vector store lived in a folder next to your notebook. That is perfect for learning and for prototypes. A product used by a whole company needs more: the store has to be reachable from many machines, survive restarts, grow past one laptop, and keep one customer's data apart from another's.

**Pinecone** is a vector database you do not run yourself. The servers are theirs; you get an API key and send documents and questions over the internet. In this notebook you load 400 articles into Pinecone and search them four ways: by meaning, with filters, by keyword, and with both combined and reranked. The last three are what production search usually looks like today.

## Setup

You need a Pinecone API key. The free plan is enough: sign up at [pinecone.io](https://www.pinecone.io), open **API keys** and copy the key. In a workshop your trainer may hand out one shared key instead; the notebook is built so that many people can use the same key at the same time without touching each other's data.

In Colab, store the key as a secret named `PINECONE_API_KEY` (the key icon on the left). Otherwise the cell asks for it.

In [1]:
%pip install -q pinecone==10.3.0 pandas   # Colab installs here; locally, `pip install -r requirements.txt` in the course folder already covers it

import os, ast, time, uuid, pathlib, urllib.request
import pandas as pd
from pinecone import Pinecone, SchemaBuilder, ConflictError

def colab_secret(name):
    """The key from Colab secrets, or None when we are not in Colab or there is no such secret."""
    try:
        from google.colab import userdata
        for attempt in range(12):                   # Colab shows a "Grant access" dialog the first time and waits only
            try:                                    # 10 seconds for your click. We keep asking for about two minutes.
                return userdata.get(name)
            except userdata.TimeoutException:
                pass
    except Exception:
        pass

def get_key(name, label):
    """Colab secret -> environment variable -> a prompt. Returns the key without stray spaces or newlines."""
    key = colab_secret(name) or os.environ.get(name)
    while not isinstance(key, str) or not key.strip():      # nothing found: type it in
        from getpass import getpass
        typed = getpass(f"{label}: ")
        key = typed if isinstance(typed, str) else colab_secret(name)   # a late click on "Grant access" lands here: read the secret again
    return key.strip()

pc = Pinecone(api_key=get_key("PINECONE_API_KEY", "Pinecone API key"))

try:                                                # one small request now, so a wrong key fails here with a clear message
    existing = [ix.name for ix in pc.indexes.list()]
except Exception as error:
    raise RuntimeError(f"Pinecone did not accept the key: {type(error).__name__}: {error}\n"
                       "Check the key (in Colab: the key icon on the left, secret PINECONE_API_KEY) and run this cell again.") from None

# Data file: next to this notebook if you downloaded the course folder; fetched from GitHub otherwise (Colab).
RAW = "https://raw.githubusercontent.com/robertbarcik/vector_databases-tutorial/main/2.%20Pinecone/"
if not pathlib.Path("articles_400.csv").exists():
    urllib.request.urlretrieve(RAW + "articles_400.csv", "articles_400.csv")

print("Connected to Pinecone. Indexes in this project:", existing or "none yet")

Note: you may need to restart the kernel to use updated packages.


Connected to Pinecone. Indexes in this project: ['vdb-course']


## 2.1 A database as a service: index, namespace, documents

### An index in the cloud

In Chroma you created a **collection** and gave it an embedding function. Pinecone's word for the same thing is an **index**, and it is described by a small **schema**: which field of your documents should be searchable, and how.

We ask for two kinds of search on the `text` field at once. `embed=` tells Pinecone to turn the text into vectors on its own servers, with a model it hosts. `full_text_search=` builds a classic keyword index over the same text. You will use both.

In [2]:
INDEX_NAME = "vdb-course"

schema = (
    SchemaBuilder()
    .add_string_field(
        "text",
        embed={"model": "llama-text-embed-v2", "dimension": 384},   # vectors are made on Pinecone's side
        full_text_search={"language": "en"},                        # and a keyword index over the same text
    )
    .build()
)

if not pc.indexes.exists(INDEX_NAME):
    try:
        pc.indexes.create(
            name=INDEX_NAME,
            schema=schema,
            deployment={"deployment_type": "managed", "cloud": "aws", "region": "us-east-1"},
        )
        print("created the index", INDEX_NAME)
    except ConflictError:                           # somebody with the same key created it a second before us
        print("the index", INDEX_NAME, "already exists")
else:
    print("the index", INDEX_NAME, "already exists, we will use it")

while not pc.indexes.describe(INDEX_NAME).status.ready:   # wait until Pinecone says it is ready
    time.sleep(2)

index = pc.index(name=INDEX_NAME)
print("index:", INDEX_NAME, "| ready:", pc.indexes.describe(INDEX_NAME).status.ready, "| searchable fields:", list(schema["fields"]))

the index vdb-course already exists, we will use it


index: vdb-course | ready: True | searchable fields: ['text']


### 🔍 What just happened?

Three lines are worth reading slowly.

`embed={"model": "llama-text-embed-v2", "dimension": 384}`: this is the embedding function from the Chroma notebooks, except that the model runs at Pinecone. You send text, they make the vector. Nothing to download, and the same model embeds your documents and your questions.

`deployment={... "cloud": "aws", "region": "us-east-1"}`: the index is **serverless**. You do not choose machines or sizes, only where it lives. The free plan offers this one region.

`if not pc.indexes.exists(...)`: an index name can exist only once per Pinecone project. If you share a key with a room full of people, the first person creates the index and everyone else finds it there. That is fine, because of what comes next.

### A namespace of your own

One index, many users: how do their documents stay apart? With a **namespace**, a named compartment inside the index. Every write and every search names exactly one namespace and never sees the others.

This is how real products separate customers. A support assistant used by 500 companies does not need 500 indexes; it needs one index with 500 namespaces, and the application picks the namespace from the logged-in customer. Here the "customer" is you.

In [3]:
# A random name, remembered for as long as this notebook session lives (re-running the cell keeps the same one).
NAMESPACE = os.environ.setdefault("VDB_NAMESPACE", "student-" + uuid.uuid4().hex[:8])
print("your namespace:", NAMESPACE)

your namespace: student-89671e27


### Loading the articles

The data: 400 articles published on Medium between 2016 and 2020, on writing, health, programming, AI and business. Each row has a title, the opening of the article, the publication year, tags and an author.

A Pinecone **document** is a plain dictionary. `_id` identifies it, `text` is the field our schema makes searchable, and everything else is stored next to it and can be used for filtering later. You do not have to declare those extra fields anywhere.

In [4]:
articles = pd.read_csv("articles_400.csv")

documents = [
    {
        "_id": f"article-{i}",
        "text": row.text,
        "title": row.title,
        "year": int(row.year),
        "tags": ast.literal_eval(row.tags),          # the CSV stores the list as text
        "author": row.author,
    }
    for i, row in articles.iterrows()
]

print(len(documents), "documents. The first one:\n")
for key, value in documents[0].items():
    print(f"  {key:7} {str(value)[:110]}")

400 documents. The first one:

  _id     article-0
  text    How to Turn Your Popular Blog Series Into a Bestselling Book

Thoughts from someone who’s done it five times


  title   How to Turn Your Popular Blog Series Into a Bestselling Book
  year    2020
  tags    ['Books', 'Entrepreneurship', 'Writing', 'Marketing', 'Productivity']
  author  Frank Mckinley


Now the upload. `upsert` means "insert, or overwrite if the id already exists", the same word Chroma uses. Pinecone accepts up to 96 documents per request when it has to embed them, so we send them in batches. The answer to each request tells us how many **tokens** the embedding model read; on the free plan that is the number to keep an eye on.

In [5]:
def my_document_count():
    for page in index.list_namespaces():
        for ns in page.namespaces:
            if ns.name == NAMESPACE:
                return ns.record_count
    return 0

if my_document_count() >= len(documents):            # already loaded in this session: do not pay for embedding twice
    print("your namespace already holds", my_document_count(), "documents, nothing to upload")
else:
    tokens = 0
    for start in range(0, len(documents), 96):
        response = index.documents.upsert(namespace=NAMESPACE, documents=documents[start:start + 96])
        tokens += response.usage.embed_total_tokens
    print(f"uploaded {len(documents)} documents, the embedding model read {tokens:,} tokens")

    while my_document_count() < len(documents):      # Pinecone indexes in the background; wait until everything is searchable
        time.sleep(2)

print("documents in your namespace:", my_document_count())

uploaded 400 documents, the embedding model read 83,634 tokens


documents in your namespace: 400


### 🔍 What just happened?

Four hundred texts went up, were embedded by `llama-text-embed-v2` on Pinecone's servers, and landed in your namespace. Compare with the Chroma notebooks: there the model was downloaded to your machine and the vectors were written to a local folder. Here your machine did no work at all, and the documents are now reachable from any program that has the key.

The free plan lets each hosted embedding model read 5 million tokens a month, so one load like this is cheap, and the cell skips the upload when your documents are already there.

### 🎯 Mini-task

Print the names of all namespaces in the index (`index.list_namespaces()`, the helper above shows how to loop over it). If you share the key with others, you will see their namespaces next to yours. Can you read their documents? Hold that thought until the first search.

In [6]:
# YOUR CODE HERE

## 2.2 Meaning or keywords? Hybrid search

### Search by meaning

The question goes in as plain text. `score_by` says how to rank: `"type": "embed"` means "embed this question with the index's model and return the documents whose vectors are closest". `include_fields` lists what we want back, so we do not download the full text of every hit.

In [7]:
def show(matches, label=""):
    if label:
        print(label)
    for m in matches:
        print(f"   {m.score:7.3f}  {int(m.year)}  {m.title[:78]}")
    if not matches:
        print("   (nothing found)")
    print()

def search_semantic(question, top_k=5, filter=None):
    return index.documents.search(
        namespace=NAMESPACE,
        top_k=top_k,
        score_by=[{"type": "embed", "fields": ["text"], "query": question}],
        include_fields=["title", "year", "tags", "text"],
        filter=filter,
    ).matches

show(search_semantic("how do I stop putting things off"), "how do I stop putting things off")

how do I stop putting things off
     0.393  2018  How to Spend The First Hour of Your Work Day on High-Value Tasks
     0.340  2020  Time Management: The Single Most Important Component to My Productivity
     0.333  2020  6 Morning Habits to Help You Seize the Day, Every Day
     0.333  2020  5 Tips on How to Use Your Unfinished Blog Posts to Help You Write New ones
     0.312  2020  Tell Your Mood to F*ck Off, Push Your Mind Harder, and Create



### 🔍 What just happened?

The question shares almost no words with those titles, and still the results are about using your time and getting things done. That is the same semantic search you built with Chroma, served from somewhere else.

One thing is the other way round. Chroma printed a **distance**, where smaller is better. Pinecone prints a **score**, here the cosine similarity, where **bigger is better**. Always check which of the two a database gives you before you set a threshold.

And the answer to the mini-task question: the search named your namespace, so it could not return anybody else's documents. Nothing to filter, nothing to forget.

### 🎯 Mini-task

Ask in your own words for articles about learning to code, about the pandemic, and about something the collection does not cover (gardening?). Watch how the top score drops when there is nothing relevant.

In [8]:
# YOUR CODE HERE

### Keyword search, and where each kind fails

Search by meaning has a blind spot: exact names. A product code, a surname, an internal project name. The embedding model has never seen your company's vocabulary, so such a word becomes a blurry vector.

Classic keyword search is the opposite. It knows nothing about meaning; it counts words (the scoring formula is called **BM25**). Our index can do both, because the schema asked for `full_text_search` too. The only change in the code is `"type": "text"`.

In [9]:
def search_keyword(question, top_k=5, filter=None):
    return index.documents.search(
        namespace=NAMESPACE,
        top_k=top_k,
        score_by=[{"type": "text", "fields": ["text"], "query": question}],
        include_fields=["title", "year", "tags", "text"],
        filter=filter,
    ).matches

for question in ["FlixStock", "how do I stop putting things off"]:
    print("=" * 100, f"\n{question}\n")
    show(search_semantic(question, top_k=3), "by meaning:")
    show(search_keyword(question, top_k=3), "by keyword:")

FlixStock



by meaning:
     0.211  2020  Video Streaming Using Flask and OpenCV
     0.210  2020  You’re more creative than you think
     0.187  2020  Dear Grocery Girl



by keyword:
     5.906  2018  How to use DeepLab in TensorFlow for object segmentation using Deep Learning

how do I stop putting things off

by meaning:
     0.393  2018  How to Spend The First Hour of Your Work Day on High-Value Tasks
     0.340  2020  Time Management: The Single Most Important Component to My Productivity
     0.333  2020  6 Morning Habits to Help You Seize the Day, Every Day



by keyword:
    11.944  2020  32 Headline Tips Every Good Writer Needs to Know
    10.402  2020  Time Management: The Single Most Important Component to My Productivity
     9.833  2018  Foxing’s ‘Nearer My God’ is the Best Song of 2018



### 🔍 What just happened?

Two questions, two different winners.

**FlixStock** is a company name that appears in exactly one article. Keyword search returns that article and nothing else. Search by meaning returns three articles that have nothing to do with it: it has no idea what "FlixStock" means, and it always returns *something*.

**"How do I stop putting things off"** goes the other way. Search by meaning finds articles about managing your time. Keyword search matches on the words "how", "stop", "things", and puts whatever repeats them most on top.

Neither is better. Questions with names and codes need keywords; questions in everyday words need meaning. Real users send both kinds, often in one sentence.

### Hybrid search: merge the two lists

So we run both searches and merge the results. This is **hybrid search**. The two lists cannot be merged by score, because the scores live on different scales (cosine similarity around 0.4, BM25 around 10). What we can compare is the **position** in each list.

**Reciprocal rank fusion** does exactly that: every document gets `1 / (60 + position)` points from each list it appears in, and the points are added up. A document near the top of either list does well; a document near the top of both wins. Pinecone runs the two searches; the merge is ten lines of Python on our side.

In [10]:
def search_hybrid(question, top_k=5, candidates=20, filter=None):
    lists = [search_semantic(question, candidates, filter), search_keyword(question, candidates, filter)]
    points, found = {}, {}
    for matches in lists:
        for position, m in enumerate(matches, start=1):
            points[m.id] = points.get(m.id, 0) + 1 / (60 + position)
            found[m.id] = m
    best = sorted(points, key=points.get, reverse=True)[:top_k]
    return [(found[i], points[i]) for i in best]

for question in ["What does FlixStock use deep learning for?", "how do I stop putting things off"]:
    print("=" * 100, f"\n{question}\n")
    show(search_semantic(question, top_k=1), "by meaning, first result:")
    show(search_keyword(question, top_k=1), "by keyword, first result:")
    print("hybrid:")
    for m, pts in search_hybrid(question, top_k=4):
        print(f"   {pts:.4f}  {int(m.year)}  {m.title[:78]}")
    print()

What does FlixStock use deep learning for?

by meaning, first result:
     0.343  2020  Video Streaming Using Flask and OpenCV



by keyword, first result:
    18.567  2018  How to use DeepLab in TensorFlow for object segmentation using Deep Learning

hybrid:


   0.0325  2018  How to use DeepLab in TensorFlow for object segmentation using Deep Learning
   0.0312  2020  WIDeText: A Multimodal Deep Learning Framework
   0.0299  2020  Video Streaming Using Flask and OpenCV
   0.0288  2019  AI analyzes language to predict schizophrenia

how do I stop putting things off

by meaning, first result:
     0.393  2018  How to Spend The First Hour of Your Work Day on High-Value Tasks



by keyword, first result:
    11.944  2020  32 Headline Tips Every Good Writer Needs to Know

hybrid:


   0.0323  2020  Time Management: The Single Most Important Component to My Productivity
   0.0299  2020  My Time-Based Challenge to Write Every Day
   0.0296  2018  How to Spend The First Hour of Your Work Day on High-Value Tasks
   0.0278  2020  Four Reasons You Won’t Make It as a Professional Writer



### 🔍 What just happened?

The first question mixes both worlds: a company name and an everyday phrase. Search by meaning alone puts a wrong article first, keyword search alone gets it right. The second question is the reverse. The hybrid list has a sensible article on top **both times**, and we did not have to guess in advance which kind of question was coming. That is the whole reason hybrid search became the default.

Look at the points. A document near the top of both lists collects about 0.03; a document that only one list found gets about half of that. The number 60 is a convention that keeps the very first position from deciding everything; you rarely need to change it.

### 🎯 Mini-task

Search for the single word `FlixStock` with `search_hybrid`. The right article now shares first place with a wrong one. Print both lists (`search_semantic`, `search_keyword`) and work out why the points come out equal. What does that tell you about merging by position when one of the two lists is pure noise?

In [11]:
# YOUR CODE HERE

## 🛠️ Developer corner: filters in Pinecone

The extra fields we stored (`year`, `tags`, `author`) work as filters straight away, without being declared in the schema. The filter language is the one you know from Chroma's `where`: `$eq`, `$gte`, `$in` and friends. Pinecone applies the filter and ranks only the documents that pass it. Both search functions above already accept a `filter`.

In [12]:
question = "how to become a better writer"

show(search_semantic(question, top_k=3), "no filter")
show(search_semantic(question, top_k=3, filter={"year": {"$lte": 2018}}), "published in 2018 or earlier")
show(search_semantic(question, top_k=3, filter={"tags": {"$in": ["Programming", "Python"]}}), "tagged Programming or Python")

no filter
     0.574  2020  The Complete Guide to Improving Your Writing Skills as a Nonnative English Spe
     0.506  2020  If You Can Give Up These 8 Things, You Can Become a Better Writer
     0.503  2020  We need 2 things to be great at what we do.

published in 2018 or earlier
     0.293  2018  The Reserve Bucket
     0.236  2018  How to Spend The First Hour of Your Work Day on High-Value Tasks
     0.215  2016  A Doooodle A Day: 9 Months, 2 Sketchbooks and 40K Words Later…



tagged Programming or Python
     0.243  2020  I Wrote a Pipeline to Publish the Best of Reddit to Instagram
     0.241  2020  3 Books to Improve Your Coding Skills
     0.190  2020  How Pets Can Make You a Better Programmer — Yes, Your Pet Can Do That



Same question, three different answers, because the pool of candidates changed. The last one is worth a second look: asked for "a better writer" among programming articles, the search returns the closest thing that pool has to offer, even if it is not very close. A filter narrows where the search looks. It does not make a weak match strong, so keep watching the scores.

**🛠️ Stretch.** Combine two conditions with `$and` (articles from 2020 tagged `AI`), and run the same filter through `search_keyword` and `search_hybrid`.

## 🛠️ Developer corner: what the free plan allows, and sharing one key

Numbers that matter when you build on Pinecone's free plan, as checked in November 2026. They change; look at Pinecone's pricing page before you plan around them.

- One project holds up to **5 indexes**, in one region. An index holds up to **100 namespaces**. That is why this notebook shares one index and gives every user a namespace, and why the clean-up cell matters.
- Each hosted embedding model reads **5 million tokens a month** for the whole project. One load of our 400 articles is about 84,000 tokens, and the load cell skips itself when your documents are already there.
- The hosted reranker answers **500 requests a month**. This notebook uses two.
- An index name exists once per project. Creating it twice raises `ConflictError`, which the first cell of section 2.1 catches, so a whole room can press Run all at the same moment.
- A schema accepts only **searchable** fields (text with `embed` or `full_text_search`, or your own vectors). Fields used for filtering are picked up from the documents automatically.
- Search by meaning and search by keyword cannot be combined inside one request, which is why the merge in section 2.2 happens in our code.

## 2.3 Reranking: a second, slower look

Embedding models and BM25 are fast because they never read the question and a document *together*. A **reranker** does: it takes the question and one candidate text, reads both, and returns one number for how well the text answers the question. That is far more accurate and far too slow to run over a whole database.

So the usual pipeline has two stages. A cheap search fetches 20 candidates, the reranker reads those 20 and keeps the best few. Pinecone hosts a reranker model too, so it is one more call.

In [13]:
def search_reranked(question, top_n=3, candidates=20):
    found = [m for m, _ in search_hybrid(question, top_k=candidates)]          # stage 1: cheap and wide
    result = pc.inference.rerank(                                              # stage 2: the reranker reads each candidate
        model="bge-reranker-v2-m3",
        query=question,
        documents=[{"id": m.id, "title": m.title, "text": m.text} for m in found],
        rank_fields=["text"],
        top_n=top_n,
        return_documents=True,
    )
    return [(r.document["title"], r.score) for r in result.data]

try:
    for question in ["What does FlixStock use deep learning for?", "how do I stop putting things off"]:
        print(question)
        for title, score in search_reranked(question):
            print(f"   {score:.3f}  {title[:78]}")
        print()
except Exception as error:                           # the free plan allows 500 rerank requests a month for the whole project
    print(f"Reranking is not available right now ({type(error).__name__}). On a shared free key the monthly allowance may be used up.")

What does FlixStock use deep learning for?


   0.903  How to use DeepLab in TensorFlow for object segmentation using Deep Learning
   0.015  WIDeText: A Multimodal Deep Learning Framework
   0.001  Beginners Guide to Cloud Computing

how do I stop putting things off


   0.054  How to Spend The First Hour of Your Work Day on High-Value Tasks
   0.010  What I Learned After 360 Days of Journaling
   0.006  How To Create An Easy Blog Calendar and Stick To It



### 🔍 What just happened?

Compare with the hybrid lists above. There the right article was first, but its points were barely ahead of the noise below it; positions say nothing about *how good* a match is.

The reranker's scores do. For the FlixStock question one article scores close to 1 and everything else close to 0: there is exactly one text here that answers it. For the second question even the best score is low. The reranker is telling you that these articles are *about* the topic but none of them answers the question directly. A RAG system can use that: pass on only what scores above some threshold, and say "I don't know" when nothing does.

Reranker scores run from 0 to 1 and mean roughly the same thing for every question, which is why a threshold works here and never quite works on raw similarity scores.

**Retrieve many cheaply, rerank a few carefully, hand the best to the model.** If you rebuild the RAG pipeline from the Chroma notebooks on a real document base, this is the retrieval step to aim for.

## 2.4 Which database when

You have now used two vector databases that sit at opposite ends: Chroma inside your notebook, Pinecone as a service on the internet. Products come and go quickly in this field, so it is more useful to remember the **four places a vector index can live** than a list of names.

- **Inside your program.** The database is a library and its data is a folder. Chroma is used this way in this course; LanceDB is another one. Nothing to install or run. The natural choice for learning, prototypes and small tools.
- **Inside the database you already have.** PostgreSQL gets vector search through the `pgvector` extension, and most established databases and search engines have added something similar. Your vectors sit next to your business data, with the same backups and the same permissions. For many companies this is the sensible first choice.
- **A dedicated vector database you run yourself.** Qdrant, Weaviate and Milvus are open-source servers built for this one job. You get the most control and the newest search features, and you operate one more system.
- **A managed service.** Pinecone, or the hosted versions of the servers above. Nothing to operate, you pay for what you use, and your data lives with the provider.

The question to ask is rarely "which one is the fastest". Ask these instead:

- **Where is the data allowed to live?** Customer contracts and regulation often decide between "inside our infrastructure" and "a service".
- **Who will operate it?** A team that already runs PostgreSQL has an answer. A team with no operations people probably wants a service.
- **How much data, and how fast does it grow?** A few million vectors fit anywhere. Hundreds of millions narrow the choice.
- **Which search features do you need?** Filters are everywhere. Hybrid search, reranking and clean separation of customers are where the products differ.

Everything you practised in this course (collections or indexes, metadata filters, chunking, hybrid search, reranking) exists in all four places under slightly different names. Moving from one to another is a matter of days, so start simple and move when you have a reason.

## Cleaning up

Delete your namespace when you are done. This removes your 400 documents and nothing else. We do **not** delete the index: on a shared key it belongs to everybody in the room.

In [14]:
index.delete_namespace(name=NAMESPACE)
print("deleted the namespace", NAMESPACE)
print("namespaces left in the index:", [ns.name for page in index.list_namespaces() for ns in page.namespaces] or "none")

deleted the namespace student-89671e27
namespaces left in the index: none


## What you take with you

- A managed vector database is the same idea as Chroma, **run by somebody else**: an index instead of a collection, an API key instead of a folder, embedding done on their side if you want.
- **Namespaces** keep users or customers apart inside one index. Every request names one namespace.
- **Filters** narrow where the search looks; they carry the business rules.
- Search by **meaning** fails on exact names and codes; search by **keyword** fails on everyday wording. **Hybrid search** runs both and merges them by position (reciprocal rank fusion).
- A **reranker** reads the question and each candidate together. Fetch 20 cheaply, keep the best 3 to 5.
- A vector index can live inside your program, inside the database you already have, in a dedicated server, or in a managed service. Choose by where the data may live and who operates it.